In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
def mae(y, z):
    '''
    Calcola l'errore medio assoluto
    '''
    return np.sum(np.abs(y-z))/y.shape[0]
def train_test_split(X, y, train_size = 0.7):
    etichette = np.unique(y)
    n = X.shape[0]
    train_idxs = []
    for lab in etichette:
        n0 = np.where(y==lab)[0].shape[0] #numero di campioni con etichetta lab
        train_idxs.append(np.random.choice(np.where(y==lab)[0], size = int(n0*train_size), replace = False))
    train_idxs = np.concatenate(train_idxs)
    test_idxs = np.setdiff1d(np.arange(n), train_idxs)
    return X[train_idxs], y[train_idxs], X[test_idxs], y[test_idxs]

Esercizio 2: modificare l'implementazione delle classi LinearRegressionGD sostituendo il criterio di terminazione dell'addestramento facendo in modo che questo termini quando la funzione J(w) smette di diminuire in modo significativo.

In [2]:
class LinearRegressionGD(object):
    def __init__(self, eta=0.001, n_iter=1000, tol=0.001):
        self.eta = eta
        self.n_iter = n_iter
        self.costi = []
        self.tol = tol
    def fit(self, X, y):
        # Standardizzazione delle feature
        # media 0, deviazione standard 1
        self.mean_ = X.mean(axis=0) # media per colonne;   COSTO Theta(nd)
        self.std_ = X.std(axis=0)   # deviazione standard per colonne; COSTO Theta(nd)
        X_std = (X - self.mean_) / self.std_ # COSTO Theta(nd)

        # Inizializzazione pesi
        self.w_ = np.zeros(1 + X.shape[1]) # COSTO Theta(d)
        for c in range(self.n_iter):  # per n_iter volte
            output = self.net_input(X_std)  # COSTO Theta(nd)
            errors = y - output  # COSTO Theta(n)
            self.w_[1:] += self.eta * X_std.T.dot(errors)/X.shape[0] # COSTO Theta(nd)
            self.w_[0] += self.eta * errors.sum()/X.shape[0] # # COSTO Theta(n)
            # divisione per X.shape[0] per maggiore stabilità
            self.costi.append(mae(y, output))
            if len(self.costi)>2 and abs(self.costi[-2]-self.costi[-1])<self.tol:
                break
        print(c)
        return self

    def net_input(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]

    def predict(self, X):
        # Standardizza i dati usando media e std del training set
        X_std = (X - self.mean_) / self.std_
        return self.net_input(X_std)
    
    def get_coef(self):
        """
        Restituisce i coefficienti e l'intercetta del modello
        riferiti ai dati originali (non standardizzati).
        """
        # Coefficienti originali
        coef = self.w_[1:] / self.std_
        
        # Intercetta originale
        intercept = self.w_[0] - np.sum(self.w_[1:] * self.mean_ / self.std_)
        
        return intercept, coef


In [3]:
df = pd.read_csv(os.path.join('dataset', 'trail_running.csv'), sep=';')
df.columns = ['Gara', 'Lunghezza', 'D+', 'Tempo']
df['Lunghezza'] = df['Lunghezza']*1000
df['Tempo'] = pd.to_timedelta(df['Tempo']).dt.total_seconds()/60
df.columns = ['Gara', 'Lunghezza', 'D+', 'Minuti']
df = df[df['Lunghezza'] < 100000]
X = df[ ['Lunghezza', 'D+'] ].values 
y = df['Minuti']
linreg = LinearRegressionGD(eta = 0.001, n_iter=2500,tol=0.001)

linreg.fit(X, y)

previsioni = linreg.predict(X)

# oppure usando i intercetta e coefficienti
intercept, coef = linreg.get_coef()
previsioni = intercept + np.dot(X, coef)

# Valori reali che il modello sta cercando di prevedere
valori_reali = y

# Calcolo del Mean Absolute Error
error = mae(valori_reali, previsioni)

print(f"Mean Absolute Error (MAE): {error:.2f}")

2499
Mean Absolute Error (MAE): 23.52


## Nota sul criterio di arresto

L'esercizio chiede di fermare l'addestramento quando la funzione $J(w)$ smette di diminuire in modo significativo.

Nel codice viene salvato:

```python
self.costi.append(mae(y, output))
```

Quindi il criterio di arresto non sta monitorando direttamente la funzione costo quadratica vista nella teoria, ma il MAE sul training set.

L'idea è comunque la stessa:

```text
se l'errore cambia pochissimo tra due iterazioni consecutive,
il modello sta migliorando poco
```

Per essere più coerenti con la notazione $J(w)$, si potrebbe usare:

```python
cost = 0.5 * np.mean(errors ** 2)
```

mentre il MAE può restare come metrica finale di valutazione.

**Esercizio 3**. Si consideri il dataset contenuto nel file `linear_regression.csv`, costituito da `2000` campioni e `11` colonne totali: le prime `10` rappresentano le feature, mentre l'ultima colonna rappresenta la variabile target `y`.

L'obiettivo dell'esercizio è addestrare un modello di regressione lineare utilizzando la classe `LinearRegressionGD`, già implementata a lezione e eventualmente estesa secondo le specifiche dell'esercizio precedente.

Dopo aver caricato il dataset dal file CSV, si richiede di separare correttamente la matrice delle feature `X` dal vettore target `y`.

Successivamente, il dataset deve essere suddiviso in un *training set* e un *test set* e addestrato sul training set.

Una volta completato l'addestramento, si richiede di valutare le prestazioni del modello calcolando il MAE e sul training set e sul test set. Infine si confrontino e valutino i risultati ottenuti.

In [4]:
df = pd.read_csv(os.path.join('dataset', 'linear_regression.csv'), sep=';',header=None)
train_size=0.7
X = df.iloc[:,:-1].values 
y = df.iloc[:, -1].values
n=X.shape[0]
train_idxs = np.random.choice(n, size = int(n*train_size), replace=False)
test_idxs =  np.setdiff1d(np.arange(n), train_idxs)

X_train, X_test = X[train_idxs], X[test_idxs]
y_train, y_test = y[train_idxs], y[test_idxs]
linreg = LinearRegressionGD(eta = 0.001, n_iter=10000,tol=0.00001)
linreg.fit(X_train, y_train)
previsioni = linreg.predict(X_train)

# oppure usando i intercetta e coefficienti
intercept, coef = linreg.get_coef()
previsioni = intercept + np.dot(X_train, coef)

# Valori reali che il modello sta cercando di prevedere
valori_reali = y_train

# Calcolo del Mean Absolute Error
error = mae(valori_reali, previsioni)

print(f"Mean Absolute Error (MAE) su train set: {error:.2f}")

6862
Mean Absolute Error (MAE) su train set: 7.88


In [5]:
previsioni = linreg.predict(X_test)
# oppure usando i intercetta e coefficienti
intercept, coef = linreg.get_coef()
previsioni = intercept + np.dot(X_test, coef)

# Valori reali che il modello sta cercando di prevedere
valori_reali = y_test

# Calcolo del Mean Absolute Error
error = mae(valori_reali, previsioni)

print(f"Mean Absolute Error (MAE) su test set: {error:.2f}")


Mean Absolute Error (MAE) su test set: 8.33


# Discesa del gradiente stocastica

Nell'algoritmo *discesa del gradiente*, ad ogni iterazione l'aggiornamento dei coefficienti viene eseguito mediante la seguente formula

$$
w \leftarrow w + \Delta w = w -\eta \nabla J(w)
$$

dove

$$
\nabla J(w) = - X^T\times \texttt{errors}.
$$

Il calcolo di $\nabla J(w)$ utilizza l'intero dataset di input ($X$) che deve essere disponibile per intero in memoria. Su dataset molto grandi questa soluzione può essere impraticabile.

**Discesa del Gradiente Stocastica**. In questa versione $\nabla J(w)$ viene calcolato usando una porzione della matrice $X$ ottenuta selezionando casualmente un numero fissato di righe (campioni). La nuova matrice ottenuta, $X_b$, corrispondente a un blocco o batch di campioni, verrà usata per calcolare $\nabla J(w)$ (e quindi i nuovi campioni) come

$$
\nabla J(w) = - X_b^T\times \texttt{errors}.
$$

In questo modo, in memoria principale si tiene soltanto $X_b$ mentre l'intero dataset può essere letto dalla memoria secondaria.  


Operativamente, durante ogni iterazione, le righe di $X$ (i campioni), vengono partizionate in *blocchi* di dimensione fissata, si costruisce $X_b$ a partire da ciascun blocco e si usa per calcolare il gradiente e aggiornare i coefficienti.

Si osservi che, mentre con Discesa del Gradiente tradizionale, ad ogni iterazione ogni coefficiente viene aggiornato solo una volta, con la Stocastica ogni coefficiente viene aggiornato tante volte quanti sono i blocchi della partizione



Se $X_b$ contiene un singolo campione, si parla di **SGD puro**, con aggiornamento dei pesi campione per campione.

## Lavoro da svolgere

Partendo da una classe `LinearRegressionGD` implementata con discesa del gradiente tradizionale, si richiede di:

1. Creare una nuova classe `LinearRegressionSGD`.

2. Modificare il metodo `fit` in modo che:

    - i campioni vengano mescolati ad ogni iterazione (shuffle);

    - il dataset venga suddiviso in blocchi di dimensione fissata;

    - per ciascun blocco, i pesi vengano aggiornati usando il gradiente calcolato solo su quel blocco;

    - a fine iterazione venga calcolata e registrata la funzione di costo media sull’intero dataset, per monitorare la convergenza.

3. Confrontare il comportamento delle due versioni utilizzando dataset sintetici generati dalla funzione `make_regression` della libreria `sklearn.datasets`. 


## Batch size e varianti della discesa del gradiente

Il parametro `batch_size` determina quanti esempi vengono usati per ogni aggiornamento dei pesi.

Ci sono tre casi principali:

```text
batch_size = n:
Gradient Descent classico.
Ogni aggiornamento usa tutto il dataset.

batch_size = 1:
SGD puro.
Ogni aggiornamento usa un solo esempio.

1 < batch_size < n:
Mini-batch Gradient Descent.
Ogni aggiornamento usa un blocco di esempi.
```

Nel codice:

```python
self.batch_size_ = X.shape[0] if self.batch_size_ == None else self.batch_size_
```

se `batch_size` non viene specificato, viene usato tutto il dataset.

Quindi la stessa classe può simulare sia la discesa del gradiente classica sia quella stocastica.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

class LinearRegressionSGD(object):
    def __init__(self, eta=0.001, n_iter=1000, tol=1e-5, batch_size=None):
        self.eta = eta
        self.n_iter = n_iter
        self.costi_ = []
        self.tol_ = tol
        self.batch_size_ = batch_size #parametro stocastico

    def fit(self, X, y):
        self.batch_size_ = X.shape[0] if self.batch_size_ == None else self.batch_size_

        # Standardizzazione delle feature
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)

        X_std = (X - self.mean_) / self.std_
        
        # Inizializzazione pesi
        self.w_ = np.zeros(1 + X.shape[1])
        
        for _ in range(self.n_iter):
            # shuffle degli indici
            shuffle_idxs = np.arange(X.shape[0])
            np.random.shuffle(shuffle_idxs)

            for b_idx in range(0, X.shape[0], self.batch_size_):#se batch_size_ è X.shape[0] funziona come l'algoritmo classico 
                X_b = X_std[shuffle_idxs[b_idx:b_idx+self.batch_size_]]
                y_b = y[shuffle_idxs[b_idx:b_idx+self.batch_size_]]

                output = self.net_input(X_b)
                errors = y_b - output
                batch_len = X_b.shape[0]
                self.w_[1:] += self.eta * X_b.T.dot(errors) / batch_len
                self.w_[0] += self.eta * errors.sum() / batch_len

            output = self.net_input(X_std)
            self.costi_.append( mae(y, output) )
            if len(self.costi_) >= 2:
                if np.abs(self.costi_[-1] - self.costi_[-2]) < self.tol_:
                    break
        return self

    def net_input(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]
        
    def predict(self, X):
        # Standardizza i dati usando media e std del training set
        X_std = (X - self.mean_) / self.std_
        
        return self.net_input(X_std)
    
    def get_coef(self):
        """
        Restituisce i coefficienti e l'intercetta del modello
        riferiti ai dati originali (non standardizzati).
        """

        # Coefficienti originali
        coef = self.w_[1:] / self.std_
        # Intercetta originale
        intercept = self.w_[0] - np.sum(self.w_[1:] * self.mean_ / self.std_)
        
        return intercept, coef
    
def mae(y, z):
    '''
    Calcola l'errore medio assoluto
    '''
    return np.sum(np.abs(y-z))/y.shape[0]

## Nota sull'ultimo batch

Nel codice l'aggiornamento divide per:

```python
self.batch_size_
```

Questo è corretto quando ogni batch contiene esattamente `batch_size_` esempi.

L'ultimo batch, però, può essere più piccolo se il numero totale di esempi non è divisibile per `batch_size`.

Una versione più robusta usa la dimensione effettiva del batch:

```python
batch_len = X_b.shape[0]
```

e divide per `batch_len`.

Nel caso dell'esempio con `2000` esempi e `batch_size=500`, il problema non si presenta perché 2000 è divisibile per 500.

In [7]:
from sklearn.datasets import make_regression

X, y = make_regression(
    n_samples=2000,       # numero di campioni
    n_features=10,        # numero di feature
    noise=10.0,          # rumore da aggiungere ai target
    random_state=42      # per riproducibilità
)

linregSGD = LinearRegressionSGD(batch_size=1, n_iter=10000, tol=0.00001)
linregSGD.fit(X, y)
errorSGD = mae(y, linregSGD.predict(X))

print(f"SGD Mean Absolute Error (MAE): {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

linregSGD = LinearRegressionSGD(batch_size=500, n_iter=10000, tol=0.00001)
linregSGD.fit(X, y)
errorSGD = mae(y, linregSGD.predict(X))

print(f"SGD Mean Absolute Error (MAE): {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

linregGD = LinearRegressionSGD(n_iter=10000, tol=0.00001)
linregGD.fit(X, y)
errorGD = mae(y, linregGD.predict(X))

print(f"GD Mean Absolute Error (MAE): {errorGD:.2f}. Iterazioni: {len(linregGD.costi_)}")



SGD Mean Absolute Error (MAE): 8.01. Iterazioni: 803
SGD Mean Absolute Error (MAE): 8.00. Iterazioni: 1936
GD Mean Absolute Error (MAE): 8.01. Iterazioni: 6608


## Interpretazione del confronto tra SGD e GD

Nel confronto vengono usati due casi:

```python
LinearRegressionSGD(batch_size=500)
```

e:

```python
LinearRegressionSGD(batch_size=None)
```

Nel primo caso il modello usa mini-batch da 500 esempi.

Nel secondo caso `batch_size` viene impostato automaticamente a tutto il dataset, quindi il comportamento corrisponde alla discesa del gradiente classica.

La differenza principale è:

```text
GD classico:
pochi aggiornamenti per epoca, ma ogni aggiornamento è più costoso

mini-batch SGD:
più aggiornamenti per epoca, ma ogni aggiornamento costa meno
```

SGD può convergere più rapidamente in termini di tempo, ma il valore della funzione costo può oscillare di più perché ogni aggiornamento usa solo una parte dei dati.

**Esercizio**. Modificare la classe `LinearRegressionSGD`introducendo una condizione di arresto basata su *validation set*. Ovvero, durante l'addestramento, l'algoritmo non deve limitarsi a controllare la convergenza sul training set, ma deve monitorare anche l'errore su un insieme di validazione fornito separatamente.

In particolare, ad ogni epoca si richiede di:

- calcolare il valore della funzione di costo sia sul training set sia sul validation set;
- mantenere in memoria il miglior valore di errore sul validation set osservato fino a quel momento
- salvare i parametri del modello (pesi e intercetta) corrispondenti a tale valore minimo.

La procedura di addestramento deve essere interrotta nel caso in cui l'errore sul validation set non migliori per un numero prefissato di epoche consecutive (parametro `patience` da aggiungere all'elenco).

Al termine dell'addestramento, il modello deve:

- ripristinare i parametri corrispondenti al miglior valore di validation error;
- restituire tali parametri come soluzione finale.

Questa modifica consente di implementare una strategia di arresto, utile a prevenire fenomeni di overfitting, in quanto l'arresto dell'algoritmo avviene nel momento in cui le prestazioni su dati non visti (validation set) smettono di migliorare, anche se l'errore sul training set continua a diminuire.

In [8]:
import matplotlib.pyplot as plt
import numpy as np
class LinearRegressionSGD(object):
    def __init__(self, eta=0.001, n_iter=1000, tol=1e-5, batch_size=None):
        self.eta = eta
        self.n_iter = n_iter
        self.costi_ = []
        self.tol_ = tol
        self.batch_size_ = batch_size

    def net_input(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]
        
    def predict(self, X):
        # Standardizza i dati usando media e std del training set
        X_std = (X - self.mean_) / self.std_
        
        return self.net_input(X_std)
    
    def get_coef(self):
        """
        Restituisce i coefficienti e l'intercetta del modello
        riferiti ai dati originali (non standardizzati).
        """

        # Coefficienti originali
        coef = self.w_[1:] / self.std_
        # Intercetta originale
        intercept = self.w_[0] - np.sum(self.w_[1:] * self.mean_ / self.std_)
        
        return intercept, coef
    
    def fit(self, X, y, X_val=None, y_val=None, patience=20):
        self.batch_size_ = X.shape[0] if self.batch_size_ is None else self.batch_size_

        # Standardizzazione (solo training)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)

        X_std = (X - self.mean_) / self.std_

        if X_val is not None:
            X_val_std = (X_val - self.mean_) / self.std_

        # Inizializzazione pesi
        self.w_ = np.zeros(1 + X.shape[1])

        # Early stopping variables
        best_val_loss = np.inf
        best_w = self.w_.copy()
        patience_count = 0

        for epoca in range(self.n_iter):

            # shuffle
            shuffle_idxs = np.arange(X.shape[0])
            np.random.shuffle(shuffle_idxs)

            for b_idx in range(0, X.shape[0], self.batch_size_):
                X_b = X_std[shuffle_idxs[b_idx:b_idx+self.batch_size_]]
                y_b = y[shuffle_idxs[b_idx:b_idx+self.batch_size_]]

                output = self.net_input(X_b)
                errors = y_b - output

                self.w_[1:] += self.eta * X_b.T.dot(errors) / self.batch_size_
                self.w_[0] += self.eta * errors.sum() / self.batch_size_

            # training loss
            output = self.net_input(X_std)
            train_loss = mae(y, output)
            self.costi_.append(train_loss)

            # --- VALIDATION ---
            if X_val is not None:
                val_output = self.net_input(X_val_std)
                val_loss = mae(y_val, val_output)

                # controllo miglioramento
                if val_loss < best_val_loss - self.tol_:
                    best_val_loss = val_loss
                    best_w = self.w_.copy()
                    patience_count = 0
                else:
                    patience_count += 1

                # condizione di arresto
                if patience_count >= patience:
                    print(f"Stop dopo {epoca} passi")
                    break
            else:
                # vecchia condizione su training
                if len(self.costi_) >= 2:
                    if np.abs(self.costi_[-1] - self.costi_[-2]) < self.tol_:
                        break

        # ripristino migliori pesi
        if X_val is not None:
            self.w_ = best_w

        return self
    
def mae(y, z):
    '''
    Calcola l'errore medio assoluto
    '''
    return np.sum(np.abs(y-z))/y.shape[0]

In [9]:
X, y = make_regression(
    n_samples=1500,       # numero di campioni
    n_features=150,        # numero di feature
    noise=5.0,          # rumore da aggiungere ai target
    random_state=42      # per riproducibilità
)

n = X.shape[0]
train_size = 0.6
val_size = 0.2
np.random.seed(10)
train_idxs = np.random.choice(n, size = int(n*train_size), replace=False)

test_val_idxs =  np.setdiff1d(np.arange(n), train_idxs)

val_idxs = np.random.choice(test_val_idxs, size = int(n*val_size), replace=False)
test_idxs = np.setdiff1d(test_val_idxs, val_idxs)

X_train, X_val, X_test = X[train_idxs], X[val_idxs], X[test_idxs]
y_train, y_val, y_test = y[train_idxs], y[val_idxs], y[test_idxs]

linregSGD = LinearRegressionSGD(batch_size=100, n_iter=10000, tol=0.001)
linregSGD.fit(X_train, y_train, X_val=X_val, y_val=y_val, patience=5)

errorSGD = mae(y_train, linregSGD.predict(X_train))
print(f"Mean Absolute Error (MAE) train set: {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

errorSGD = mae(y_test, linregSGD.predict(X_test))
print(f"Mean Absolute Error (MAE) test set: {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

Stop dopo 912 passi
Mean Absolute Error (MAE) train set: 3.86. Iterazioni: 913
Mean Absolute Error (MAE) test set: 4.50. Iterazioni: 913


## Perché usare un validation set per l'early stopping

Il criterio precedente controllava solo l'errore sul training set.

Questo può essere limitante, perché il training error tende spesso a diminuire anche quando il modello inizia a generalizzare peggio.

Con il validation set, invece, controlliamo le prestazioni su dati non usati direttamente per aggiornare i pesi.

La variabile:

```python
best_val_loss
```

memorizza il miglior errore di validazione osservato.

La variabile:

```python
best_w
```

salva i pesi corrispondenti a quel miglior errore.

Il parametro:

```python
patience
```

indica per quante epoche consecutive accettiamo di non migliorare prima di fermare il training.

Alla fine, il modello ripristina i pesi migliori:

```python
self.w_ = best_w
```

In questo modo la soluzione finale non è necessariamente quella dell'ultima epoca, ma quella che ha funzionato meglio sul validation set.

## Nota sullo split nei problemi di regressione

La funzione `train_test_split` definita all'inizio è adatta a problemi di classificazione, perché divide i dati preservando la distribuzione delle classi.

Nei problemi di regressione, però, il target `y` è continuo.

Quindi non ha senso usare direttamente:

```python
np.unique(y)
```

come se i valori di `y` fossero classi.

Per questa lezione è più adatto lo split casuale tramite indici, come fatto nell'esercizio su `linear_regression.csv`.